# 12 — vLLM: LongBench Benchmark

This notebook evaluates KV cache compression on
[LongBench](https://github.com/THUDM/LongBench) tasks using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B.

We test two compression algorithms (**full_replacement**, **filtering**) at
multiple compression ratios on two LongBench tasks:
- **gov_report** — summarization (high decoding stress, long generated output)
- **hotpotqa** — multi-hop QA (multi-document reasoning)

Scoring uses HuggingFace `evaluate`:
- ROUGE-L for gov_report (summarization)
- F1 for hotpotqa (QA)

Results are saved to `results/vllm_longbench/` for comparison in later notebooks.

## Configuration

In [1]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

FRACTION = 0.01

LONGBENCH_TASKS = ["gov_report", "hotpotqa"]

MAX_NEW_TOKENS = {
    "gov_report": 512,
    "hotpotqa": 64,
}

PRESS_CONFIGS = {
    "full_replacement": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False,
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [2]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


Using FORK vLLM (version: dev)
Using FORK vLLM (version: dev)


In [4]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n\u26a0  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

GPU:        NVIDIA A100-SXM4-40GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       42.4 GB (approx)
GPU:        NVIDIA A100-SXM4-40GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       42.4 GB (approx)


## 1. Load LongBench Datasets

In [5]:
from datasets import load_dataset

longbench_datasets = {}
for task_name in LONGBENCH_TASKS:
    ds = load_dataset("THUDM/LongBench", task_name, split="test")
    if FRACTION < 1.0:
        n = max(1, int(len(ds) * FRACTION))
        ds = ds.select(range(n))
    longbench_datasets[task_name] = ds
    print(f"{task_name}: {len(ds)} examples")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


gov_report: 2 examples
gov_report: 2 examples
hotpotqa: 2 examples
hotpotqa: 2 examples


## 2. Load Scoring Metrics

In [6]:
import evaluate

rouge_metric = evaluate.load("rouge")
squad_metric = evaluate.load("squad")

TASK_METRICS = {
    "gov_report": "rouge",
    "hotpotqa": "squad",
}

print("Loaded scoring metrics: rouge (gov_report), squad F1 (hotpotqa)")

Loaded scoring metrics: rouge (gov_report), squad F1 (hotpotqa)
Loaded scoring metrics: rouge (gov_report), squad F1 (hotpotqa)


## 3. Prepare Prompts

Apply the model's chat template to each LongBench example.

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = {}
max_context_tokens = 0

for task_name in LONGBENCH_TASKS:
    ds = longbench_datasets[task_name]
    task_prompts = []

    for row in ds:
        user_msg = row["context"] + "\n\n" + row["input"]
        messages = [{"role": "user", "content": user_msg}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

        n_tokens = len(tokenizer.encode(prompt, add_special_tokens=False))
        max_context_tokens = max(max_context_tokens, n_tokens)

        task_prompts.append({
            "prompt": prompt,
            "task": task_name,
            "answers": row["answers"],
        })

    prompt_configs[task_name] = task_prompts
    print(f"{task_name}: {len(task_prompts)} prompts prepared")

print(f"\nMax context tokens: {max_context_tokens}")

gov_report: 2 prompts prepared
gov_report: 2 prompts prepared
hotpotqa: 2 prompts prepared

Max context tokens: 17247
hotpotqa: 2 prompts prepared

Max context tokens: 17247


## 4. Run Batch Inference

vLLM processes all prompts per task in a single batch via its internal
scheduler. A new LLM instance is created per (algorithm, compression_ratio)
combination.

In [8]:
import time
from vllm import LLM, SamplingParams

def get_gpu_memory_used_gb() -> float:
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

sampling_params_per_task = {
    task: SamplingParams(temperature=0.0, max_tokens=max_tok)
    for task, max_tok in MAX_NEW_TOKENS.items()
}

configs = [("no_press", 0.0)]
for press_name in PRESS_CONFIGS:
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio))

all_results = []

for press_name, ratio in configs:
    llm = None
    try:
        if press_name == "no_press":
            llm = LLM(
                model=MODEL_NAME,
                dtype="auto",
                gpu_memory_utilization=0.90,
                trust_remote_code=True,
                attention_config={"backend": "FLASH_ATTN"},
            )
        else:
            llm_kwargs = PRESS_CONFIGS[press_name](ratio)
            llm = LLM(**llm_kwargs)

        for task_name in LONGBENCH_TASKS:
            prompts_list = prompt_configs[task_name]
            prompts = [pc["prompt"] for pc in prompts_list]
            sp = sampling_params_per_task[task_name]

            label = f"{press_name} | ratio={ratio} | {task_name}"
            print(f"\n{'='*60}")
            print(f"Running: {label} ({len(prompts)} examples)")
            print(f"{'='*60}")

            mem_before = get_gpu_memory_used_gb()
            start = time.perf_counter()

            outputs = llm.generate(prompts, sp)

            batch_elapsed = time.perf_counter() - start
            mem_after = get_gpu_memory_used_gb()
            peak_mem = max(mem_before, mem_after)

            for i, output in enumerate(outputs):
                predicted_answer = output.outputs[0].text.strip()
                pc = prompts_list[i]

                all_results.append({
                    "framework": "vllm",
                    "press": press_name,
                    "compression_ratio": ratio,
                    "task": task_name,
                    "answers": pc["answers"],
                    "predicted_answer": predicted_answer,
                    "elapsed_sec": round(batch_elapsed / len(prompts), 3),
                    "peak_gpu_mem_gb": round(peak_mem, 3),
                })

            total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
            throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0
            print(f"  Done: {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")

    finally:
        if llm is not None:
            cleanup_vllm(llm)

print(f"\nTotal results: {len(all_results)}")

INFO 08-31 15:25:59 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.FLASH_ATTN: 'vllm.v1.attention.backends.flash_attn.FlashAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'model': 'Qwen/Qwen3-8B'}
INFO 08-31 15:26:00 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 08-31 15:26:00 [model.py:1582] Using max model len 40960
INFO 08-31 15:26:00 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-31 15:26:00 [vllm.py:795] Asynchronous scheduling is enabled.
WARNING 08-31 15:26:02 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overr

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


(EngineCore pid=117400) INFO 08-31 15:26:10 [core.py:103] Initializing a V1 LLM engine (vdev) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_c

(EngineCore pid=117400) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=117400) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:00<00:02,  1.38it/s]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:01<00:02,  1.00it/s]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:03<00:02,  1.09s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:04<00:01,  1.05s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:04<00:00,  1.23it/s]
Loading safetensors checkpoint shards: 100% Complet

(EngineCore pid=117400) INFO 08-31 15:26:18 [default_loader.py:384] Loading weights took 4.52 seconds
(EngineCore pid=117400) INFO 08-31 15:26:19 [gpu_model_runner.py:4627] Model loading took 15.27 GiB memory and 5.513396 seconds
(EngineCore pid=117400) INFO 08-31 15:26:23 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/f1781885f0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=117400) INFO 08-31 15:26:23 [backends.py:1048] Dynamo bytecode transform time: 4.47 s
(EngineCore pid=117400) INFO 08-31 15:26:25 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.305 s
(EngineCore pid=117400) INFO 08-31 15:26:25 [monitor.py:48] torch.compile took 6.13 s in total
(EngineCore pid=117400) INFO 08-31 15:26:25 [decorators.py:296] Directly load AOT compilation from path /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/b5b87fb51ca98babdcdb46b974078c98a405315eff9cadba9be7225f8

...

(EngineCore pid=119806) INFO 08-31 15:35:14 [gpu_model_runner.py:5807] Graph capturing finished in 5 secs, took 0.52 GiB
(EngineCore pid=119806) INFO 08-31 15:35:14 [gpu_worker.py:617] CUDA graph pool memory: 0.52 GiB (actual), 0.52 GiB (estimated), difference: 0.0 GiB (0.8%).
(EngineCore pid=119806) INFO 08-31 15:35:14 [core.py:281] init engine (profile, create kv cache, warmup model) took 14.11 seconds
(EngineCore pid=119806) INFO 08-31 15:35:15 [vllm.py:795] Asynchronous scheduling is enabled.
INFO 08-31 15:35:15 [llm.py:391] Supported tasks: ['generate']

Running: filtering | ratio=0.75 | gov_report (2 examples)

Running: filtering | ratio=0.75 | gov_report (2 examples)


Processed prompts: 100%|██████████| 2/2 [00:44<00:00, 22.12s/it, est. speed input: 628.90 toks/s, output: 23.15 toks/s]


  Done: 44.3s — 23.1 tok/s — peak mem=40.19 GB
  Done: 44.3s — 23.1 tok/s — peak mem=40.19 GB
Running: filtering | ratio=0.75 | hotpotqa (2 examples)


Running: filtering | ratio=0.75 | hotpotqa (2 examples)


Processed prompts: 100%|██████████| 2/2 [00:06<00:00,  3.37s/it, est. speed input: 4301.87 toks/s, output: 19.01 toks/s]
[rank0]:[W831 15:36:06.324083099 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


  Done: 6.8s — 18.9 tok/s — peak mem=40.19 GB
  Done: 6.8s — 18.9 tok/s — peak mem=40.19 GB
(EngineCore pid=119806) INFO 08-31 15:36:06 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=119806) INFO 08-31 15:36:06 [core.py:1224] Shutdown complete

Total results: 36

Total results: 36


## 5. Score & Results

Score predictions using HuggingFace `evaluate`:
- ROUGE-L for gov_report (summarization)
- F1 for hotpotqa (QA, via SQuAD metric)

In [9]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []

for (press, ratio, task), group in df.groupby(
    ["press", "compression_ratio", "task"]
):
    preds = group["predicted_answer"].tolist()
    refs = group["answers"].tolist()
    key = f"{press}__{ratio}__{task}"

    if TASK_METRICS[task] == "rouge":
        result = rouge_metric.compute(
            predictions=preds,
            references=[r[0] if isinstance(r, list) else r for r in refs],
        )
        score = round(result["rougeL"] * 100, 2)
        metric_name = "rougeL"
    else:
        squad_preds = [{"id": str(i), "prediction_text": p} for i, p in enumerate(preds)]
        squad_refs = [{"id": str(i), "answers": {"text": r if isinstance(r, list) else [r], "answer_start": [0] * (len(r) if isinstance(r, list) else 1)}} for i, r in enumerate(refs)]
        result = squad_metric.compute(predictions=squad_preds, references=squad_refs)
        score = round(result["f1"], 2)
        metric_name = "f1"

    all_metrics[key] = {metric_name: score}
    rows.append({
        "press": press, "compression_ratio": ratio,
        "task": task, "metric": metric_name, "score": score,
        "mean_time": round(group["elapsed_sec"].mean(), 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

           press  compression_ratio       task metric  score  mean_time
       filtering               0.01 gov_report rougeL  21.29     29.068
       filtering               0.01   hotpotqa     f1   5.88      4.825
       filtering               0.25 gov_report rougeL  21.21     26.580
       filtering               0.25   hotpotqa     f1   6.38      4.462
       filtering               0.50 gov_report rougeL  20.47     21.475
       filtering               0.50   hotpotqa     f1   6.38      3.742
       filtering               0.75 gov_report rougeL  18.37     22.149
       filtering               0.75   hotpotqa     f1   0.00      3.394
full_replacement               0.01 gov_report rougeL  22.23      5.441
full_replacement               0.01   hotpotqa     f1   5.88      1.903
full_replacement               0.25 gov_report rougeL  20.03      5.219
full_replacement               0.25   hotpotqa     f1   6.52      1.806
full_replacement               0.50 gov_report rougeL  19.60    

## 6. Save Results

In [10]:
import json

os.makedirs("results/vllm_longbench", exist_ok=True)

predictions_path = "results/vllm_longbench/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm_longbench/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/vllm_longbench/predictions.csv
Saved predictions to results/vllm_longbench/predictions.csv
Saved metrics to results/vllm_longbench/metrics.json
Saved metrics to results/vllm_longbench/metrics.json
